In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [4]:
from scipy.integrate import quad
from scipy.special import spherical_jn

In [1]:
import sys
sys.path.append("../src/")
from interpolation import RDP_subsample

In [3]:
"""
Minimal usage example: importing the Eq(33)-based Hankel transform
pipeline to compute int f(x) * j_l(kx) * 4*pi*x^2 dx over a given
x-interval, for a simple, smooth (non-oscillatory) test function f(x).

Note: no u, v coefficient table needed here (unlike the Rayleigh
method) -- Eq(33) only needs x, freqs, and l_values.

Run directly (`python example_usage_eq33.py`) for a quick sanity check
against scipy's own spherical Bessel / quadrature routines.
"""

import numpy as np
from scipy.integrate import quad
from scipy.special import spherical_jn

from single_bessel_recurrence import precompute_dI_eq33, hankel_transform_multi_l_eq33


# 1. A simple, smooth (non-oscillatory) test function.
def f(x):
    return np.exp(-x**2 / 4.0)


# 2. Sample grids: 200 x-points over [0.5, 8], 50 frequencies over [0.2, 3].
x = np.linspace(0.5, 8.0, 2000)
freqs = np.linspace(0.2, 3.0, 50)
l_values = [0, 5, 23]  # any l values at once -- no cutoff to worry about

fx_weighted = 4 * np.pi * x**2 * f(x)  # the usual r^2-weighted convention

# 3. Precompute (once per x, freqs, l_values -- reusable across many f's).
dI0, dI1 = precompute_dI_eq33(x, freqs, l_values)

# 4. Evaluate for this particular f (the only per-f step).
result = hankel_transform_multi_l_eq33(x, fx_weighted, freqs, l_values, dI0, dI1)
# result has shape (len(l_values), len(freqs))

if __name__ == "__main__":
    print("l    k          our result       scipy quad       abs diff")
    for li, l in enumerate(l_values):
        for ki in [0, 25, 49]:
            k = freqs[ki]
            true_val = quad(lambda xx: 4*np.pi*xx**2*f(xx)*spherical_jn(l, k*xx),
                             x.min(), x.max(), limit=200)[0]
            approx = result[li, ki]
            print(f"{l:<3}  {k:6.3f}  {approx: .10f}  {true_val: .10f}  {abs(approx-true_val):.2e}")

l    k          our result       scipy quad       abs diff
0     0.200   42.2959836121   42.2960172961  3.37e-05
0     1.629   2.6682524325   2.6682506201  1.81e-06
0     3.000  -0.3948702148  -0.3948740120  3.80e-06
5     0.200   0.0002898548   0.0002898543  5.58e-10
5     1.629   2.3331511282   2.3331501562  9.72e-07
5     3.000   3.2936850428   3.2936888383  3.80e-06
23    0.200   0.0000000000   0.0000000000  1.22e-34
23    1.629   0.0000000029   0.0000000029  2.88e-14
23    3.000   0.0001984988   0.0001984878  1.10e-08


In [4]:
%timeit hankel_transform_multi_l_eq33(x, fx_weighted, freqs, l_values, dI0, dI1)

1.17 ms ± 983 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
